# Week 2: In-class warm-up problems

**Course:** AI for Aerospace & Mechanical Engineering (PhD), Block 0
**Format:** 4 problems, 5 to 10 minutes each, pen and paper first, then verify in Python.

> **Provenance:** new content. Class/instance attributes and `super()` behaviour rewritten
> from [W3Schools Class Properties](https://www.w3schools.com/python/python_class_properties.asp),
> [GeeksforGeeks class vs instance variables](https://www.geeksforgeeks.org/python/g-fact-34-class-or-static-variables-in-python/)
> and [GeeksforGeeks `super()`](https://www.geeksforgeeks.org/python/python-super/);
> ISA pressure altitude adapted from Anderson, *Introduction to Flight*, Ch. 3.

## Problem 1 (5 min): whose list is it?

<div dir="rtl">مسئله ۱: بدون اجرای کد، خروجی هر سطر <code>print</code> را پیش‌بینی کنید و توضیح دهید چرا دو ژیروسکوپ داده‌های یکدیگر را «می‌بینند».</div>

```python
class Gyro:
    unit = "deg/s"
    samples = []

    def __init__(self, name):
        self.name = name
        self.bias = 0.0

    def add(self, x):
        self.samples.append(x - self.bias)

g1, g2 = Gyro("g1"), Gyro("g2")
g1.bias = 0.5
g1.add(2.0)
g2.add(1.0)
g2.unit = "rad/s"
print(g1.samples, g2.samples)
print(g1.unit, g2.unit, Gyro.unit)
```

a) Predict both printed lines.
b) Why does `g2.unit = "rad/s"` not change `g1.unit`, while `g1.add` changes `g2.samples`?
c) Fix the class with a one-line change.

## Problem 2 (10 min): follow `super()` through three classes

<div dir="rtl">مسئله ۲: ترتیب چاپ پیام‌های سازنده‌ها و خروجی <code>describe()</code> را پیش‌بینی کنید. اگر یکی از کلاس‌ها <code>super().__init__</code> را صدا نزند چه می‌شود؟</div>

```python
class Sensor:
    def __init__(self, sid):
        print("Sensor init")
        self.sid = sid

    def describe(self):
        return self.sid

class InertialSensor(Sensor):
    def __init__(self, sid, rate):
        print("Inertial init")
        super().__init__(sid)
        self.rate = rate

    def describe(self):
        return f"inertial {super().describe()} @ {self.rate:g} Hz"

class IMU(InertialSensor):
    def __init__(self, sid):
        print("IMU init")
        super().__init__(sid, 200.0)

    def describe(self):
        return "IMU/" + super().describe()

imu = IMU("imu0")
print(imu.describe())
```

a) In which order are the three `init` messages printed?
b) What does `imu.describe()` return?
c) Write `IMU.__mro__` as a list of class names.
d) If `InertialSensor.__init__` forgot the `super().__init__(sid)` line, which line would fail,
   and with which error?

## Problem 3 (10 min): calibrate, preprocess, validate an IMU by hand

<div dir="rtl">مسئله ۳: از چهار نمونه‌ی حالت سکون، بایاس شتاب‌سنج را حساب کنید؛ سپس یک نمونه‌ی پروازی را پیش‌پردازش کرده، ضریب بار آن را بیابید و درباره‌ی اعتبار نمونه‌ی دوم تصمیم بگیرید.</div>

An IMU set to **±2 g** (full scale $2 \times 9.80665$ m/s²) gives four accelerometer samples
while parked level (m/s²):

| # | $a_x$ | $a_y$ | $a_z$ |
|---|------:|------:|------:|
| 1 | 0.04 | −0.02 | 9.90 |
| 2 | 0.06 | −0.03 | 9.86 |
| 3 | 0.05 | −0.01 | 9.88 |
| 4 | 0.05 | −0.02 | 9.92 |

a) The true specific force at rest is $(0, 0, +g_0)$. Compute the bias vector.
b) In flight the raw sample $(0.55, -0.02, 17.08)$ arrives. Preprocess it and compute the
   load factor $n$.
c) A later raw sample has $a_z = 19.70$ m/s². What should `validate` return, and what does
   that tell the structures engineer about the true acceleration?

## Problem 4 (10 min): barometric climb rate, two ways

<div dir="rtl">مسئله ۴: دو فشار با فاصله‌ی ۱ ثانیه داده شده است. ارتفاع هر کدام را با مدل ISA حساب کنید و نرخ صعود را بیابید؛ سپس با رابطه‌ی هیدرواستاتیک تقریبی مقایسه کنید.</div>

A barometer (reference 101 325 Pa) reads 89 875 Pa, and 1.0 s later 89 760 Pa.

a) Using $h = 44\,330.8\,[1 - (p/101\,325)^{0.190263}]$, compute both altitudes and the
   climb rate.
b) Check with the hydrostatic relation $\Delta h \approx -\Delta p / (\rho g_0)$, with the
   ISA density at 1000 m, $\rho = p/(R T)$, $T = 288.15 - 0.0065 h$, $R = 287.05$ J/(kg·K).
c) The barometer's noise is 0.8 m (one sigma). Is a 1-s difference of two raw samples a good
   climb-rate estimate? What would you do instead?

In [ ]:
import numpy as np

# Scratch space for verifying your hand calculations.